In [1]:
print("hello")

hello


In [2]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] += r";C:\hadoop\bin"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("StatefulProcessingNotebook") \
    .master("local[*]") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
    ) \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("Spark session started")

Spark session started


In [4]:
order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("event_time", StringType(), True)
])

print("Schema created")

Schema created


In [5]:
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "127.0.0.1:9092") \
    .option("subscribe", "order_events") \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

print("Kafka stream connected")

Kafka stream connected


In [6]:
events_df = df.selectExpr(
    "CAST(value AS STRING) as json_data"
).select(
    from_json(col("json_data"), order_schema).alias("order_data")
).select(
    col("order_data.order_id").alias("order_id"),
    col("order_data.customer_id").alias("customer_id"),
    col("order_data.product_id").alias("product_id"),
    col("order_data.event_type").alias("event_type"),
    col("order_data.quantity").alias("quantity"),
    col("order_data.price").alias("price"),
    to_timestamp(col("order_data.event_time")).alias("event_timestamp")
)

print("Event-time dataframe ready")

Event-time dataframe ready


In [7]:
raw_events_query = events_df.writeStream \
    .format("memory") \
    .queryName("raw_events_check") \
    .outputMode("append") \
    .start()

print("Raw events validation stream started")

Raw events validation stream started


In [8]:
raw_events_query.awaitTermination(15)

False

In [13]:
spark.sql("""
SELECT 
    order_id,
    product_id,
    event_type,
    event_timestamp,
    COUNT(*) AS duplicate_count
FROM raw_events_check
GROUP BY order_id, product_id, event_type, event_timestamp
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC
""").show(truncate=False)

+--------+----------+----------+--------------------------+---------------+
|order_id|product_id|event_type|event_timestamp           |duplicate_count|
+--------+----------+----------+--------------------------+---------------+
|order_4 |prod_10   |CREATED   |2026-03-31 11:26:21.354988|2              |
|order_30|prod_60   |CANCELLED |2026-03-31 11:24:16.446698|2              |
|order_28|prod_25   |CREATED   |2026-03-31 11:25:30.485169|2              |
|order_12|prod_64   |UPDATED   |2026-03-31 11:26:22.522585|2              |
|order_30|prod_35   |CREATED   |2026-03-31 11:25:41.668998|2              |
|order_16|prod_17   |CREATED   |2026-03-31 11:26:16.230464|2              |
|order_31|prod_77   |CREATED   |2026-03-31 11:24:55.192673|2              |
|order_31|prod_12   |CREATED   |2026-03-31 11:26:27.841811|2              |
|order_19|prod_55   |CREATED   |2026-03-31 11:26:12.529345|2              |
|order_7 |prod_43   |CREATED   |2026-03-31 11:24:41.906231|2              |
|order_19|pr

In [14]:
spark.sql("""
SELECT *
FROM (
    SELECT
        order_id,
        product_id,
        event_type,
        event_timestamp,
        COUNT(*) OVER (
            PARTITION BY order_id, product_id, event_type, event_timestamp
        ) AS duplicate_count
    FROM raw_events_check
) t
WHERE duplicate_count > 1
ORDER BY event_timestamp DESC
LIMIT 20
""").show(truncate=False)

+--------+----------+----------+--------------------------+---------------+
|order_id|product_id|event_type|event_timestamp           |duplicate_count|
+--------+----------+----------+--------------------------+---------------+
|order_18|prod_49   |CREATED   |2026-03-31 11:31:43.588077|2              |
|order_18|prod_49   |CREATED   |2026-03-31 11:31:43.588077|2              |
|order_42|prod_45   |CREATED   |2026-03-31 11:31:40.765951|2              |
|order_42|prod_45   |CREATED   |2026-03-31 11:31:40.765951|2              |
|order_32|prod_48   |CREATED   |2026-03-31 11:31:39.933267|2              |
|order_32|prod_48   |CREATED   |2026-03-31 11:31:39.933267|2              |
|order_22|prod_7    |CREATED   |2026-03-31 11:31:24.72239 |2              |
|order_22|prod_7    |CREATED   |2026-03-31 11:31:24.72239 |2              |
|order_9 |prod_6    |CANCELLED |2026-03-31 11:31:19.957828|2              |
|order_9 |prod_6    |CANCELLED |2026-03-31 11:31:19.957828|2              |
|order_16|pr

In [15]:
dedup_df = events_df \
    .withWatermark("event_timestamp", "10 minutes") \
    .dropDuplicates([
        "order_id",
        "product_id",
        "event_type",
        "event_timestamp"
    ])

print("Watermark and deduplication applied")

Watermark and deduplication applied


In [16]:
dedup_query = dedup_df.writeStream \
    .format("memory") \
    .queryName("dedup_orders_check") \
    .outputMode("append") \
    .start()

print("Dedup validation stream started")

Dedup validation stream started


In [17]:
dedup_query.awaitTermination(10)

False

In [21]:
spark.sql("""
SELECT *
FROM (
    SELECT
        order_id,
        product_id,
        event_type,
        event_timestamp,
        COUNT(*) OVER (
            PARTITION BY order_id, product_id, event_type, event_timestamp
        ) AS duplicate_count
    FROM dedup_orders_check
) t
WHERE duplicate_count =1
LIMIT 20
""").show(truncate=False)

+--------+----------+----------+--------------------------+---------------+
|order_id|product_id|event_type|event_timestamp           |duplicate_count|
+--------+----------+----------+--------------------------+---------------+
|order_1 |prod_15   |CREATED   |2026-03-31 11:56:30.618432|1              |
|order_1 |prod_18   |CREATED   |2026-03-31 11:47:49.056256|1              |
|order_1 |prod_31   |CREATED   |2026-03-31 11:53:50.337602|1              |
|order_1 |prod_36   |CANCELLED |2026-03-31 11:54:02.820232|1              |
|order_1 |prod_40   |CREATED   |2026-03-31 11:50:24.13698 |1              |
|order_1 |prod_43   |UPDATED   |2026-03-31 11:46:00.905047|1              |
|order_1 |prod_44   |CANCELLED |2026-03-31 11:45:11.268369|1              |
|order_1 |prod_47   |CREATED   |2026-03-31 11:43:40.053446|1              |
|order_1 |prod_50   |CANCELLED |2026-03-31 11:51:19.709719|1              |
|order_1 |prod_52   |CREATED   |2026-03-31 11:45:47.60999 |1              |
|order_1 |pr

In [31]:
spark.sql("""
SELECT *
FROM dedup_orders_check
WHERE order_id = 'order_1'
ORDER BY event_timestamp DESC
LIMIT 1
""").show(truncate=False)

+--------+-----------+----------+----------+--------+-----+--------------------------+
|order_id|customer_id|product_id|event_type|quantity|price|event_timestamp           |
+--------+-----------+----------+----------+--------+-----+--------------------------+
|order_1 |cust_14    |prod_71   |CREATED   |2       |76.42|2026-03-31 12:07:12.825172|
+--------+-----------+----------+----------+--------+-----+--------------------------+



In [ ]:

# Get the latest event (current state) for each order_id using event-time ordering

spark.sql("""
SELECT *
FROM (
    SELECT
        order_id,
        customer_id,
        product_id,
        event_type,
        event_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY event_timestamp DESC
        ) AS rn
    FROM dedup_orders_check
) t
WHERE rn = 1 and order_id = 'order_1'
LIMIT 20
""").show(truncate=False)

+--------+-----------+----------+----------+--------------------------+---+
|order_id|customer_id|product_id|event_type|event_timestamp           |rn |
+--------+-----------+----------+----------+--------------------------+---+
|order_1 |cust_14    |prod_71   |CREATED   |2026-03-31 12:07:12.825172|1  |
+--------+-----------+----------+----------+--------------------------+---+

